In [1]:
import os
# 选择GPU
os.environ['CUDA_VISIBLE_DEVICES'] = '4'
os.chdir('/home/zheling/Character_Recognition')

print("当前工作目录：", os.getcwd())


当前工作目录： /home/zheling/Character_Recognition


In [2]:
import math
import time
import subprocess
import array
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from typing import List
from tqdm import tqdm
from joblib import Parallel, delayed
from esig import tosig


In [3]:
def get_gpu_memory_usage_mb():
    try:
        if not torch.cuda.is_available():
            return 0
        result = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,nounits,noheader'],
            encoding='utf-8'
        )
        mem_values = [int(x) for x in result.strip().split('\n')]
        current_device = torch.cuda.current_device()
        return mem_values[current_device]
    except Exception:
        return 0
        

In [4]:
class DyadicPathSignatures:
    def __init__(self, dyadic_levels: int, signature_level: int = 2, overlapping: bool = False) -> None:
        self._dyadic_levels = dyadic_levels
        self._signature_level = signature_level
        self._overlapping = overlapping

    def __call__(self, sample: np.ndarray) -> np.ndarray:
        dyadic_pieces: List[List[np.ndarray]] = [[] for i in range(sample.shape[0])]
        for dyadic_level in range(self._dyadic_levels + 1):
            if self._overlapping:
                num_pieces = 2**(dyadic_level + 1) - 1
                frames_per_piece = sample.shape[1] / (num_pieces - 1) if num_pieces > 1 else sample.shape[1]
            else:
                num_pieces = 2**dyadic_level
                frames_per_piece = sample.shape[1] / num_pieces

            for i in range(sample.shape[0]):
                for j in range(num_pieces):
                    start_frame = int(j * frames_per_piece / 2**self._overlapping)
                    end_frame = int(start_frame + frames_per_piece)
                    dyadic_pieces[i].append(
                        tosig.stream2sig(
                            sample[i][start_frame:end_frame].reshape(end_frame - start_frame, -1).astype(np.float64),
                            self._signature_level
                        )
                    )
        return np.array(dyadic_pieces)

    def get_description(self) -> dict:
        return {
            "(t)DySig/dylvl": self._dyadic_levels,
            "(t)DySig/siglvl": self._signature_level,
            "(t)DySig/overlap": self._overlapping
        }
        

In [5]:
def hanging_normalization(input_array, method='sc'):
    trajectory = input_array[:, :2]
    T = trajectory.shape[0]
    start_point = trajectory[0]
    center_point = trajectory.mean(axis=0) if method == 'sc' else trajectory[-1]
    
    dx = center_point[0] - start_point[0]
    dy = center_point[1] - start_point[1]
    angle = np.arctan2(dy, dx) + np.pi / 2

    normalized_trajectory = np.zeros_like(trajectory)
    cos_a, sin_a = np.cos(-angle), np.sin(-angle)
    for t in range(T):
        x, y = trajectory[t]
        normalized_trajectory[t] = [
            (x - start_point[0]) * cos_a - (y - start_point[1]) * sin_a,
            (x - start_point[0]) * sin_a + (y - start_point[1]) * cos_a
        ]
    
    output = input_array.copy()
    output[:, :2] = normalized_trajectory
    return output

def normalize_stroke(input_array, is_keep_ratio=True):
    stroke = input_array[:, :2].copy()
    min_x, min_y = stroke.min(axis=0)
    max_x, max_y = stroke.max(axis=0)
    width, height = max_x - min_x, max_y - min_y
    
    if is_keep_ratio:
        factor = max(width, height)
        if factor > 0:
            stroke[:, 0] = (stroke[:, 0] - min_x) / factor
            stroke[:, 1] = (stroke[:, 1] - min_y) / factor
    else:
        if width > 0:
            stroke[:, 0] = (stroke[:, 0] - min_x) / width
        if height > 0:
            stroke[:, 1] = (stroke[:, 1] - min_y) / height
    
    output = input_array.copy()
    output[:, :2] = stroke
    return output

def apply_rotation(trajectory, angle_degrees):
    angle_rad = np.radians(angle_degrees)
    cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
    rotation_matrix = np.array([[cos_a, -sin_a], [sin_a, cos_a]])
    result = trajectory.copy()
    result[:, :2] = trajectory[:, :2] @ rotation_matrix.T
    return result

def add_imaginary_strokes(strokes, enable_imaginary=True):
    if not enable_imaginary:
        return strokes
    
    total_segments, total_length = 0, 0
    for stroke in strokes:
        if len(stroke) > 1:
            lengths = np.linalg.norm(stroke[1:, :2] - stroke[:-1, :2], axis=1)
            total_length += np.sum(lengths)
            total_segments += len(lengths)
    
    if total_segments == 0:
        return strokes
    
    avg_segment_length = total_length / total_segments
    result_strokes = []
    
    for i in range(len(strokes)):
        result_strokes.append(strokes[i])
        if i < len(strokes) - 1:
            start_point = strokes[i][-1]
            end_point = strokes[i+1][0]
            virtual_length = np.linalg.norm(end_point[:2] - start_point[:2])
            num_segments = max(1, int(np.ceil(virtual_length / avg_segment_length)))
            
            virtual_stroke = []
            for j in range(num_segments + 1):
                alpha = j / num_segments
                virtual_point = start_point.copy()
                virtual_point[:2] = start_point[:2] * (1-alpha) + end_point[:2] * alpha
                virtual_stroke.append(virtual_point)
            result_strokes.append(np.array(virtual_stroke))
    
    return result_strokes

def read_pot_file(filename, isFlipY=True):
    a = array.array('H')
    with open(filename, 'rb') as f:
        a.fromfile(f, os.path.getsize(filename) // a.itemsize)
    a.reverse()
    samples = []
    
    while len(a) > 0:
        a.pop()
        a.pop()
        a.pop()
        num_strokes = a.pop()
        strokes = []
        
        for _ in range(num_strokes):
            stroke = []
            point = [a.pop(), a.pop()]
            while point != [65535, 0]:
                if isFlipY:
                    point = [point[0], -point[1]]
                stroke.append(point)
                point = [a.pop(), a.pop()]
            strokes.append(np.array(stroke, dtype=np.float32))
        a.pop()
        a.pop()
        samples.append(strokes)
    
    return samples

def interpolate_1d(x, target_length):
    old_length = len(x)
    old_indices = np.linspace(0, old_length - 1, old_length)
    new_indices = np.linspace(0, old_length - 1, target_length)
    return np.interp(new_indices, old_indices, x)

def interpolate_trajectory(trajectory, target_length):
    result = np.zeros((target_length, trajectory.shape[1]))
    for i in range(trajectory.shape[1]):
        result[:, i] = interpolate_1d(trajectory[:, i], target_length)
    return result

def process_class(class_idx, data_dir, sig, min_npoint, angle, is_train=True,
                   concat_strokes=False, hang_normalize=True, is_add_ink_dim=True,
                   enable_imaginary=True, normalize_ink=True, master_seed=42):
    filename = os.path.join(data_dir, f"{class_idx}.pot")
    if not os.path.exists(filename):
        return np.empty((0, 0)), np.empty(0, dtype=np.int64)
    
    raw_samples = read_pot_file(filename)
    features, labels = [], []
    N = len(raw_samples)
    train_border = int(0.8 * N)
    samples_to_process = raw_samples[:train_border] if is_train else raw_samples[train_border:]

    for strokes in samples_to_process:
        strokes = [apply_rotation(st, angle) for st in strokes]
        
        if is_add_ink_dim:
            ink_released = 0
            for i, stroke in enumerate(strokes):
                ink_dim = np.arange(ink_released, ink_released + stroke.shape[0], dtype=np.float32)
                ink_released += stroke.shape[0] - 1
                strokes[i] = np.concatenate([stroke, ink_dim[:, None]], axis=-1)
        
        strokes_imaginary = add_imaginary_strokes(strokes, enable_imaginary) if enable_imaginary else []

        def feature_extraction(stk_list):
            if is_add_ink_dim and normalize_ink:
                final_ink = max([stroke[-1, -1] for stroke in stk_list if len(stroke) > 0])
                if final_ink > 0:
                    for i in range(len(stk_list)):
                        stk_list[i] = stk_list[i].copy()
                        stk_list[i][:, -1] = stk_list[i][:, -1] / final_ink
            
            sample = np.concatenate(stk_list, axis=0) if concat_strokes and stk_list else (stk_list[0] if stk_list else np.empty((0, 3)))
            if sample.size == 0:
                return None
            
            onetraj = normalize_stroke(sample)
            target_len = min_npoint * onetraj.shape[0]
            onetraj = interpolate_trajectory(onetraj, target_len)
            
            normalized_onetraj = hanging_normalization(onetraj, method='sc') if hang_normalize else onetraj
            normalized_onetraj = normalize_stroke(normalized_onetraj)
            
            
            fea = sig(normalized_onetraj[None, ...])
            fea = fea[..., 1:].flatten()
            return fea

        feature = feature_extraction(strokes)
        if feature is None:
            continue
        
        if enable_imaginary and strokes_imaginary:
            feature_imaginary = feature_extraction(strokes_imaginary)
            if feature_imaginary is not None:
                feature = np.concatenate([feature, feature_imaginary], axis=-1)
                
        features.append(feature)
        labels.append(class_idx - 1)

    return (np.stack(features) if features else np.empty((0, 0)),
            np.array(labels) if labels else np.empty(0, dtype=np.int64))

class RadicalDataset:
    def __init__(self, 
                 data_dir,
                 num_classes,
                 concat_strokes=False,
                 hang_normalize=True,
                 is_add_ink_dim=True,
                 is_train=True,
                 feature_norm_factor=None,
                 enable_imaginary=True,
                 normalize_ink=True,
                 master_seed=42,
                 dyadic_levels=3,
                 signature_level=3,
                 overlapping=False,
                 min_point=None,
                 is_parallel=True):
        
        self.feature_norm_factor = feature_norm_factor
        self.sig = DyadicPathSignatures(
            dyadic_levels=dyadic_levels,
            signature_level=signature_level,
            overlapping=overlapping
        )
        
        self.min_point = min_point if min_point is not None else 2 ** dyadic_levels + 1
        
        if is_train:
            np.random.seed(master_seed)
            tasks = [(cls, np.random.uniform(0, 360)) for cls in range(1, num_classes + 1)]
        else:
            tasks = [(cls, angle_idx * 12) for cls in range(1, num_classes + 1) for angle_idx in range(30)]
        
        print(f"{'Train dataset' if is_train else 'Test dataset'}: Totally {len(tasks)} tasks")
        
        if is_parallel:
            results = Parallel(n_jobs=-1)(
                delayed(process_class)(
                    cls, data_dir, self.sig, self.min_point, ang, is_train,
                    concat_strokes, hang_normalize, is_add_ink_dim,
                    enable_imaginary, normalize_ink, master_seed
                )
                for cls, ang in tqdm(tasks, desc="Processing")
            )
        else:
            results = []
            for cls, ang in tqdm(tasks, desc="Processing"):
                res = process_class(
                    cls, data_dir, self.sig, self.min_point, ang, is_train,
                    concat_strokes, hang_normalize, is_add_ink_dim,
                    enable_imaginary, normalize_ink, master_seed
                )
                results.append(res)
        
        valid_results = [(f, l) for f, l in results if len(f) > 0]
        if valid_results:
            self.features = np.concatenate([f for f, _ in valid_results], axis=0)
            self.labels = np.concatenate([l for _, l in valid_results], axis=0)
        else:
            self.features = np.empty((0, 0))
            self.labels = np.empty(0, dtype=np.int64)
        
        if len(self.features) > 0:
            self.feature_normalization()
            print(f"{'Train dataset' if is_train else 'Test dataset'} Features shape: {self.features.shape}, Labels shape: {self.labels.shape}")

    def feature_dim(self):
        return self.features.shape[1] if len(self.features) > 0 else 0

    def feature_normalization(self):
        if self.feature_norm_factor is None:
            max_val = np.max(np.abs(self.features), axis=0)
            max_val[max_val == 0] = 1
            self.feature_norm_factor = max_val
        self.features = self.features / self.feature_norm_factor



In [6]:
class RotationFreeOLHCR(nn.Module):
    def __init__(self, input_size, num_classes):
        super(RotationFreeOLHCR, self).__init__()
        self.layers = nn.ModuleList([
            nn.Linear(input_size, 128),
            nn.Linear(128, 256),
            nn.Linear(256, 384),
            nn.Linear(384, 512),
            nn.Linear(512, 640)
        ])
        self.dropouts = nn.ModuleList([nn.Dropout(p=p) for p in [0.1, 0.2, 0.3, 0.4, 0.5]])
        self.output_layer = nn.Linear(640, num_classes)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        for i in range(5):
            x = self.layers[i](x)
            x = self.relu(x)
            x = self.dropouts[i](x)
        return self.output_layer(x)
        

In [7]:
from thop import profile, clever_format

def get_model_metrics(model, input_dim, batch_size=1):
    device = next(model.parameters()).device
    dummy_input = torch.randn(batch_size, input_dim).to(device)
    macs, params = profile(model, inputs=(dummy_input,), verbose=False)
    flops = macs * 2.0
    flops_str, params_str = clever_format([flops, params], "%.3f")
    return params_str, flops_str

    
def train_step(model, X, y, optimizer, device):
    model.train()
    X, y = X.to(device), y.to(device)
    
    optimizer.zero_grad()
    logits = model(X)
    loss = nn.CrossEntropyLoss()(logits, y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    with torch.no_grad():
        acc = (logits.argmax(dim=1) == y).float().mean()
    
    return loss.item(), acc.item()

def train_epoch(model, loader, optimizer, device):
    losses, accs = [], []
    for X, y in loader:
        loss, acc = train_step(model, X, y, optimizer, device)
        losses.append(loss)
        accs.append(acc)
    return np.mean(losses), np.mean(accs), len(loader)

def evaluate_accuracy(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total



def train_loop(model, train_loader, test_loader, epochs, lr, device):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    log = {"epoch": [], "loss": [], "acc": [], "test_acc": [], "time": [], "mem": []}
    total_steps, t0_total = 0, time.time()

    for ep in range(epochs):
        t0_epoch = time.time()
        l, a, steps_in_epoch = train_epoch(model, train_loader, optimizer, device)
        total_steps += steps_in_epoch
        cur_mem = get_gpu_memory_usage_mb()
        epoch_time = time.time() - t0_epoch
        
        test_acc = evaluate_accuracy(model, test_loader, device)

        if (ep+1) % 10 == 0 or ep == 0:
            print(f"Epoch {ep+1}/{epochs} | Loss: {l:.4f} | Train Acc: {a:.2%} | Test Acc: {test_acc:.2%} | Mem: {cur_mem:.2f}MB | Time: {epoch_time:.2f}s")

        log["epoch"].append(ep+1)
        log["loss"].append(l)
        log["acc"].append(a)
        log["test_acc"].append(test_acc)
        log["time"].append(epoch_time)
        log["mem"].append(cur_mem)
    
    total_time = time.time() - t0_total
    avg_mem = float(np.mean(log["mem"])) if log["mem"] else 0.0
    time_per_1000_steps = (total_time / total_steps * 1000) if total_steps > 0 else 0.0

    stats = {
        "total_time": total_time,
        "total_steps": total_steps,
        "avg_mem": avg_mem,
        "time_per_1000_steps": time_per_1000_steps
    }
    return model, log, stats

In [8]:
def run_single_model(train_loader, test_loader, input_dim, num_classes, seed, save_dir, device, num_epochs=300, lr=1e-3):
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    model = RotationFreeOLHCR(input_dim, num_classes).to(device)

    params_str, flops_str = get_model_metrics(model, input_dim=input_dim)
    print(f"\n[模型分析] RFOLHCR 参数量: {params_str}, FLOPs: {flops_str}")

    model, log, stats = train_loop(model, train_loader, test_loader, num_epochs, lr, device)

    test_acc = evaluate_accuracy(model, test_loader, device)

    os.makedirs(save_dir, exist_ok=True)
    epochs_to_save = min(100, len(log.get('test_acc', [])))
    if epochs_to_save > 0:
        df_acc = pd.DataFrame({
            "Epoch": list(range(1, epochs_to_save + 1)),
            "Test_Accuracy": log['test_acc'][:epochs_to_save]
        })
        csv_path = os.path.join(save_dir, f"test_acc_first_100_epochs_seed_{seed}.csv")
        df_acc.to_csv(csv_path, index=False)
        print(f"前 {epochs_to_save} 个 Epoch 的测试精度已保存至: {csv_path}")

    print(f"Time per 1000 steps: {stats['time_per_1000_steps']:.2f}s")
    print(f"Avg GPU Memory: {stats['avg_mem']:.2f}MB")
    print(f"[Seed {seed}] Final Test Accuracy: {test_acc*100:.2f}%")
    
    return model, test_acc


def ensemble_evaluate(models, test_loader, device):
    all_logits = []
    all_targets = None

    for model in models:
        model.eval()
        logits_list, targets_list = [], []
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                logits = model(X)
                logits_list.append(logits.cpu().numpy())
                targets_list.append(y.cpu().numpy())

        logits_all = np.concatenate(logits_list, axis=0)
        targets_all = np.concatenate(targets_list, axis=0)

        all_logits.append(logits_all)
        if all_targets is None:
            all_targets = targets_all

    all_logits = np.stack(all_logits, axis=0)  # [S, N, C]
    targets = all_targets

    # soft voting
    probs = np.exp(all_logits) / np.sum(np.exp(all_logits), axis=-1, keepdims=True)
    avg_probs = np.mean(probs, axis=0)
    soft_acc = np.mean(np.argmax(avg_probs, -1) == targets)

    # hard voting
    seed_preds = np.argmax(all_logits, axis=-1)  # [S, N]
    hard_preds = []
    for i in range(seed_preds.shape[1]):
        votes = np.bincount(seed_preds[:, i])
        hard_preds.append(np.argmax(votes))
    hard_preds = np.array(hard_preds)
    hard_acc = np.mean(hard_preds == targets)

    return soft_acc, hard_acc

def get_dataloader(data_dir, num_classes, batch_size=32, is_parallel=True, **kwargs):
    dataset = RadicalDataset(data_dir, num_classes, is_parallel=is_parallel, **kwargs)
    X = torch.from_numpy(dataset.features).float()
    y = torch.from_numpy(dataset.labels).long()
    
    if kwargs.get("is_train", True):
        perm = torch.randperm(len(y))
        X, y = X[perm], y[perm]
    
    tensor_dataset = TensorDataset(X, y)
    loader = DataLoader(tensor_dataset, batch_size=batch_size, shuffle=kwargs.get("is_train", True))
    
    return loader, dataset.feature_dim(), dataset.feature_norm_factor



In [ ]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}\n")
    
    seed_list = [1011, 1012, 1013, 1014, 1015]
    num_classes = 10
    data_dir = "RotFreeData/Digits10"  # EngUpper26 / Digits10 /Radicals52
    SAVE_DIR = "Model/RotateFreeOLHCR/RotateFreeOLHCR_Digits10"  
    MASTER_SEED = 42
    batch_size = 64

    print("Building training data...")
    train_loader, train_feadim, train_norm_factor = get_dataloader(
        data_dir, num_classes, batch_size=batch_size,
        concat_strokes=True, hang_normalize=True,
        is_add_ink_dim=True, is_train=True,
        feature_norm_factor=None,
        enable_imaginary=True, normalize_ink=True,
        master_seed=MASTER_SEED,
        dyadic_levels=3,       
        signature_level=4,      
        overlapping=False,
        is_parallel=True
    )

    print("Building test data...")
    test_loader, test_feadim, _ = get_dataloader(
        data_dir, num_classes, batch_size=batch_size,
        concat_strokes=True, hang_normalize=True,
        is_add_ink_dim=True, is_train=False,
        feature_norm_factor=train_norm_factor,
        enable_imaginary=True, normalize_ink=True,
        master_seed=MASTER_SEED,
        dyadic_levels=3,       
        signature_level=4,      
        overlapping=False,
        is_parallel=True
    )

    assert train_feadim == test_feadim
    print(f"\nFeature dimension: {train_feadim}")
    print(f"Training samples: {len(train_loader.dataset)}")
    print(f"Test samples: {len(test_loader.dataset)}\n")

    models, test_accs = [], []
    for seed in seed_list:
        print(f"\n===== Seed {seed} training started =====\n")
        model, acc = run_single_model(
            train_loader, test_loader, train_feadim, num_classes,
            seed=seed, save_dir=SAVE_DIR, device=device, num_epochs=300
        )
        models.append(model)
        test_accs.append(acc)

    mean_acc = np.mean(test_accs)
    std_acc = np.std(test_accs)
    print("\n===== Single-Model Accuracy Statistics =====")
    print(f"Mean Test Accuracy: {mean_acc*100:.2f}%")
    print(f"Std Dev: {std_acc*100:.2f}%")
    print(f"Mean ± Std: {mean_acc*100:.2f}% ± {std_acc*100:.2f}%")

    soft_acc, hard_acc = ensemble_evaluate(models, test_loader, device)
    print("\n===== Ensemble Evaluation =====")
    print(f"Soft voting accuracy: {soft_acc*100:.2f}%")
    print(f"Hard voting accuracy: {hard_acc*100:.2f}%")